# Enron Document Entity Extraction — Part 2
**Pipeline:** parse emails → preprocessing → word embeddings (FastText) → weak labels → gold set → CRF baseline → Transformer (Hugging Face) → entity-level evaluation → error analysis → MLflow

Entity types: `PER` (Person), `ORG` (Organization), `LOC` (Location), `DATE`, `PROD` (Product), `REF` (Reference number)

In [ ]:
%pip install -q gensim sklearn-crfsuite seqeval mlflow transformers datasets accelerate torch

In [ ]:
import re, os, json, random, email
from collections import Counter
import numpy as np, pandas as pd, spacy, kagglehub

random.seed(42); np.random.seed(42)
N_EMAILS = 5000          # زوديها لو جهازك يستحمل
nlp = spacy.load("en_core_web_sm")
ENT_TYPES = ["PER", "ORG", "LOC", "DATE", "PROD", "REF"]

## 1) Load a *random* sample of emails
أول 1000 صف في الداتا كلهم من mailbox واحد (`allen-p`)، فالموديل هيتعلم على شخص واحد بس. هنا بناخد عينة عشوائية من كل الملف.

In [ ]:
path = kagglehub.dataset_download("wcukierski/enron-email-dataset")
csv_path = os.path.join(path, "emails.csv")

parts = [c.sample(min(len(c), 500), random_state=42) for c in pd.read_csv(csv_path, chunksize=20000)]
df = pd.concat(parts).sample(frac=1, random_state=42).head(N_EMAILS).reset_index(drop=True)
print(df.shape, df["file"].str.split("/").str[0].nunique(), "mailboxes")

## 2) Preprocessing
- نفصل الـ **headers** عن الـ **body** (الـ NER بيشتغل على الـ body بس)
- نشيل الـ forwarded / original message المتكرر
- **مهم:** ما بنعملش lowercase ولا حذف punctuation على النص اللي رايح للـ NER (الـ capitalization بتفرق جداً في Person/Org). الـ normalization (lowercase, numbers → `<num>`) بتتعمل على نسخة منفصلة للـ embeddings بس.

In [ ]:
CUT = re.compile(r"-{2,}\s*(?:Original Message|Forwarded by)|^\s*From:.*\n\s*Sent:", re.I | re.M)

def parse_email(raw):
    msg = email.message_from_string(raw)
    body = msg.get_payload()
    body = body if isinstance(body, str) else ""
    m = CUT.search(body)
    if m: body = body[:m.start()]
    return {"from": msg["From"], "to": msg["To"], "subject": msg["Subject"],
            "date": msg["Date"], "x_from": msg["X-From"], "body": body}

def clean_body(text):
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

parsed = pd.DataFrame([parse_email(m) for m in df["message"]])
parsed["body"] = parsed["body"].apply(clean_body).str.slice(0, 3000)
df = pd.concat([df[["file"]], parsed], axis=1)
df = df[df["body"].str.len() > 50].reset_index(drop=True)
print(len(df), "emails after filtering")
df[["subject", "body"]].head(3)

In [ ]:
# Tokenization + POS + lemma (spaCy)  — الـ doc بيحتوي كل ده
docs = list(nlp.pipe(df["body"], batch_size=32))

def norm_tok(t):                     # text normalization (للـ embeddings بس)
    if t.like_email: return "<email>"
    if t.like_num or any(c.isdigit() for c in t.text): return "<num>"
    return t.lower_

d = docs[0]
print([(t.text, t.pos_, t.lemma_, norm_tok(t)) for t in d[:15]])

## 3) Weak (silver) labels
مفيش labels جاهزة في Enron، فبنعمل silver labels بقواعد:
`REF` regex → `DATE` regex → `PROD` gazetteer → spaCy pretrained (PERSON/ORG/GPE/LOC/DATE/PRODUCT).
الـ silver فيه أخطاء، عشان كده التقييم الحقيقي بيكون على **gold set** بتصححيه بإيدك (القسم 5).

In [ ]:
from spacy.matcher import PhraseMatcher

REF_PATTERNS = [re.compile(p, re.I) for p in [
    r"\b(?:deal|ticket|ref(?:erence)?|po|p\.o\.|invoice|inv|confirmation|conf|contract|order|tracking|case|id|account|acct)\s*(?:no\.?|number|num|#)?\s*[:#-]?\s*([A-Z]*\d[\w\-/]{3,})",
    r"\b([A-Z]{2,5}[-_]\d{4,})\b",
    r"#\s*(\d{4,})",
]]
DATE_PATTERNS = [re.compile(p) for p in [
    r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
    r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2}(?:st|nd|rd|th)?(?:,?\s+\d{4})?\b",
]]
PRODUCT_TERMS = ["natural gas", "enrononline", "enron online", "eol", "lotus notes", "excel", "outlook",
                 "windows nt", "internet explorer", "powerpoint", "sitara", "enpower", "gas daily", "unify"]  # وسّعيها من الداتا
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher.add("PROD", [nlp.make_doc(t) for t in PRODUCT_TERMS])
SPACY2OURS = {"PERSON": "PER", "ORG": "ORG", "GPE": "LOC", "LOC": "LOC", "DATE": "DATE", "PRODUCT": "PROD"}

def weak_label(doc):
    cands = []
    for p in REF_PATTERNS:
        for m in p.finditer(doc.text):
            cands.append((0, *m.span(1), "REF"))
    for p in DATE_PATTERNS:
        for m in p.finditer(doc.text):
            cands.append((1, *m.span(), "DATE"))
    for _, s, e in matcher(doc):
        cands.append((2, doc[s].idx, doc[e - 1].idx + len(doc[e - 1]), "PROD"))
    for ent in doc.ents:
        if ent.label_ in SPACY2OURS:
            cands.append((3, ent.start_char, ent.end_char, SPACY2OURS[ent.label_]))
    cands.sort(key=lambda x: (x[0], x[1]))
    taken, out = [], []
    for _, s, e, l in cands:
        if any(s < te and e > ts for ts, te in taken): continue
        taken.append((s, e)); out.append((s, e, l))
    return out

def sent_to_rec(sent):
    toks = [t for t in sent if not t.is_space]
    return {"idx": [t.i for t in toks], "tokens": [t.text for t in toks],
            "pos": [t.pos_ for t in toks], "norm": [norm_tok(t) for t in toks]}

def build_records(docs):
    records = []
    for eid, doc in enumerate(docs):
        tags_all = ["O"] * len(doc)
        for s, e, l in weak_label(doc):
            sp = doc.char_span(s, e, alignment_mode="expand")
            if sp is None or any(tags_all[i] != "O" for i in range(sp.start, sp.end)): continue
            tags_all[sp.start] = "B-" + l
            for i in range(sp.start + 1, sp.end): tags_all[i] = "I-" + l
        split = "train" if eid % 10 < 8 else ("dev" if eid % 10 == 8 else "test")   # split by EMAIL (no leakage)
        for sent in doc.sents:
            r = sent_to_rec(sent)
            if not 3 <= len(r["tokens"]) <= 100: continue
            r["tags"] = [tags_all[i] for i in r["idx"]]
            if r["tags"][0].startswith("I-"): r["tags"][0] = "B-" + r["tags"][0][2:]
            r.update(eid=eid, split=split)
            records.append(r)
    return records

records = build_records(docs)
print(len(records), "sentences")
print(Counter(t[2:] for r in records for t in r["tags"] if t.startswith("B-")))

## 4) Word embeddings (FastText)
بنتدرب unsupervised على نص كل الإيميلات (من غير labels، فمفيش leakage). FastText أحسن من Word2Vec هنا لأن فيه كلمات غريبة/misspelled كتير ولازم يتعامل مع الـ OOV.

In [ ]:
from gensim.models import FastText
EMB_DIM = 25
ft = FastText([r["norm"] for r in records], vector_size=EMB_DIM, window=5, min_count=3, sg=1, epochs=10, workers=4, seed=42)
for w in ["gas", "meeting", "houston"]:
    print(w, "->", [x for x, _ in ft.wv.most_similar(w, topn=5)])

## 5) Gold set (يدوي — مهم جداً)
هنصدّر ~200 جملة من الـ test split بالـ silver labels، وانتي بتصححي عمود `tag` بس (في Excel / Google Sheets / LibreOffice) وتحفظيه باسم `gold.csv`.
الـ tags بصيغة BIO مثل `B-PER`, `I-PER`, `O`. التقييم النهائي بيتعمل على الملف ده.

In [ ]:
rng = random.Random(42)
test_recs = [r for r in records if r["split"] == "test"]
has_ent = [r for r in test_recs if any(t != "O" for t in r["tags"])]
no_ent = [r for r in test_recs if all(t == "O" for t in r["tags"])]
to_fix = rng.sample(has_ent, min(150, len(has_ent))) + rng.sample(no_ent, min(50, len(no_ent)))

rows = [{"sent_id": k, "token": t, "pos": p, "norm": n, "tag": g}
        for k, r in enumerate(to_fix) for t, p, n, g in zip(r["tokens"], r["pos"], r["norm"], r["tags"])]
pd.DataFrame(rows).to_csv("gold_to_fix.csv", index=False, encoding="utf-8-sig")
print("saved gold_to_fix.csv  -> صححي عمود tag واحفظيه كـ gold.csv")

In [ ]:
def load_gold(path="gold.csv"):
    g = pd.read_csv(path, dtype=str, keep_default_na=False)
    return [{"tokens": d["token"].tolist(), "pos": d["pos"].tolist(), "norm": d["norm"].tolist(), "tags": d["tag"].tolist()}
            for _, d in g.groupby("sent_id", sort=False)]

if os.path.exists("gold.csv"):
    gold = load_gold(); print("gold sentences:", len(gold))
else:
    gold = [{k: r[k] for k in ("tokens", "pos", "norm", "tags")} for r in to_fix]
    print("⚠️ gold.csv مش موجود — بستخدم silver مؤقتاً. النتائج دي مش للتسليم!")

train = [r for r in records if r["split"] == "train"]
dev = [r for r in records if r["split"] == "dev"]
# نحتفظ بكل الجمل اللي فيها entities + 20% من الباقي (توفير ذاكرة وتوازن)
MAX_TRAIN = 8000
pos_s = [r for r in train if any(t != "O" for t in r["tags"])]
neg_s = [r for r in train if all(t == "O" for t in r["tags"])]
train_small = (pos_s + rng.sample(neg_s, min(len(neg_s), len(pos_s) // 4)))
rng.shuffle(train_small); train_small = train_small[:MAX_TRAIN]
print(len(train_small), "train |", len(dev), "dev |", len(gold), "gold")

## 6) Baseline: CRF + hand-crafted features + FastText embeddings (POS كمان feature)

In [ ]:
import sklearn_crfsuite, mlflow
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

def shape(w):
    s = re.sub(r"[A-Z]", "X", re.sub(r"[a-z]", "x", re.sub(r"\d", "d", w)))
    return re.sub(r"(.)\1+", r"\1\1", s)

def tok_feats(r, i, prefix=""):
    w = r["tokens"][i]
    return {prefix + "lower": w.lower(), prefix + "title": w.istitle(), prefix + "upper": w.isupper(),
            prefix + "shape": shape(w), prefix + "pos": r["pos"][i]}

def word2features(r, i):
    w = r["tokens"][i]
    f = {"bias": 1.0, "suf3": w[-3:], "pre3": w[:3], "isdigit": w.isdigit(),
         "hasdigit": any(c.isdigit() for c in w), "hasdash": "-" in w, "hasat": "@" in w, **tok_feats(r, i)}
    for k, v in enumerate(ft.wv[r["norm"][i]]): f[f"emb{k}"] = float(v)
    n = len(r["tokens"])
    for off in (-2, -1, 1, 2):
        j = i + off
        if 0 <= j < n: f.update(tok_feats(r, j, f"{off}:"))
    if i == 0: f["BOS"] = True
    if i == n - 1: f["EOS"] = True
    return f

def sent2features(r): return [word2features(r, i) for i in range(len(r["tokens"]))]

X_train = [sent2features(r) for r in train_small]; y_train = [r["tags"] for r in train_small]
crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True)
crf.fit(X_train, y_train)

In [ ]:
def evaluate(y_true, y_pred):
    return classification_report(y_true, y_pred, output_dict=True, zero_division=0)

def log_report(rep, prefix):
    for k, v in rep.items():
        for m in ("precision", "recall", "f1-score"):
            mlflow.log_metric(f"{prefix}_{k.replace(' ', '_')}_{m.replace('-', '_')}", v[m])

def predict_crf(recs): return crf.predict([sent2features(r) for r in recs])

mlflow.set_experiment("enron-ner")
with mlflow.start_run(run_name="crf_fasttext"):
    mlflow.log_params({"model": "CRF", "emb": "FastText", "emb_dim": EMB_DIM, "c1": 0.1, "c2": 0.1,
                       "n_train_sents": len(train_small), "n_gold_sents": len(gold)})
    pred_dev = predict_crf(dev); pred_gold = predict_crf(gold)
    rep_dev = evaluate([r["tags"] for r in dev], pred_dev)
    rep_gold = evaluate([r["tags"] for r in gold], pred_gold)
    log_report(rep_dev, "silver_dev"); log_report(rep_gold, "gold")

print("=== Entity-level on GOLD ==="); print(classification_report([r["tags"] for r in gold], pred_gold, zero_division=0))

## 7) Optional extension: Transformer NER (Hugging Face)
بنعمل fine-tune لـ `distilbert-base-cased` على نفس الـ labels (لو عندك GPU/Colab/Kaggle استخدمي `bert-base-cased` أو `dslim/bert-base-NER`).

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)

MODEL_NAME = "distilbert-base-cased"
LABELS = ["O"] + [f"{p}-{l}" for l in ENT_TYPES for p in ("B", "I")]
l2i = {l: i for i, l in enumerate(LABELS)}
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(batch):
    enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=128)
    enc["labels"] = []
    for b, tags in enumerate(batch["tags"]):
        prev, lab = None, []
        for w in enc.word_ids(batch_index=b):
            lab.append(-100 if (w is None or w == prev) else l2i[tags[w]])
            prev = w
        enc["labels"].append(lab)
    return enc

def to_ds(recs):
    return Dataset.from_list([{"tokens": r["tokens"], "tags": r["tags"]} for r in recs]).map(
        encode, batched=True, remove_columns=["tokens", "tags"])

ds_train, ds_dev, ds_gold = to_ds(train_small), to_ds(dev[:1500]), to_ds(gold)

def decode(preds, labels):
    yp, yt = [], []
    for p_row, l_row in zip(preds, labels):
        yp.append([LABELS[p] for p, l in zip(p_row, l_row) if l != -100])
        yt.append([LABELS[l] for l in l_row if l != -100])
    return yt, yp

def compute_metrics(ev):
    yt, yp = decode(np.argmax(ev.predictions, -1), ev.label_ids)
    return {"f1": f1_score(yt, yp), "precision": precision_score(yt, yp), "recall": recall_score(yt, yp)}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=l2i)
args = TrainingArguments(output_dir="ner_model", eval_strategy="epoch", save_strategy="no", learning_rate=5e-5,
                         per_device_train_batch_size=16, num_train_epochs=3, weight_decay=0.01,
                         logging_steps=50, report_to="none", seed=42)
trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_dev,
                  data_collator=DataCollatorForTokenClassification(tok), compute_metrics=compute_metrics,
                  processing_class=tok)

with mlflow.start_run(run_name=f"transformer_{MODEL_NAME}"):
    mlflow.log_params({"model": MODEL_NAME, "epochs": 3, "lr": 5e-5, "batch": 16, "n_train_sents": len(train_small)})
    trainer.train()
    out = trainer.predict(ds_gold)
    yt_g, yp_g = decode(np.argmax(out.predictions, -1), out.label_ids)
    # لو في جملة اتقطعت بسبب max_length نكمّل بـ O عشان الأطوال تتطابق مع gold
    pred_gold_bert = [yp + ["O"] * (len(r["tags"]) - len(yp)) for yp, r in zip(yp_g, gold)]
    rep_gold_bert = evaluate([r["tags"] for r in gold], pred_gold_bert)
    log_report(rep_gold_bert, "gold")

print(classification_report([r["tags"] for r in gold], pred_gold_bert, zero_division=0))

## 8) Comparison (entity-level F1 on gold)

In [ ]:
cmp = pd.DataFrame({"CRF+FastText": {k: v["f1-score"] for k, v in rep_gold.items()},
                    "Transformer": {k: v["f1-score"] for k, v in rep_gold_bert.items()}}).round(3)
cmp

## 9) Error analysis
بنصنّف الأخطاء: **missed** (FN) / **spurious** (FP) / **wrong_type** / **boundary** (نفس النوع بس حدود غلط).

In [ ]:
from seqeval.metrics.sequence_labeling import get_entities

def error_analysis(recs, preds, model_name):
    rows = []
    for r, yp in zip(recs, preds):
        gt, pr = set(get_entities(r["tags"])), set(get_entities(list(yp)))
        ov = lambda a, b: a[1] <= b[2] and b[1] <= a[2]
        for g in gt - pr:
            hits = [p for p in pr - gt if ov(g, p)]
            if not hits: kind = "missed"
            elif any((p[1], p[2]) == (g[1], g[2]) for p in hits): kind = "wrong_type"
            elif any(p[0] == g[0] for p in hits): kind = "boundary"
            else: kind = "boundary+type"
            rows.append({"model": model_name, "error": kind, "type": g[0], "gold": " ".join(r["tokens"][g[1]:g[2]+1]),
                         "pred": " | ".join(f"{p[0]}:{' '.join(r['tokens'][p[1]:p[2]+1])}" for p in hits),
                         "sentence": " ".join(r["tokens"])})
        for p in pr - gt:
            if not any(ov(p, g) for g in gt):
                rows.append({"model": model_name, "error": "spurious", "type": p[0], "gold": "",
                             "pred": " ".join(r["tokens"][p[1]:p[2]+1]), "sentence": " ".join(r["tokens"])})
    return pd.DataFrame(rows)

errs = pd.concat([error_analysis(gold, pred_gold, "CRF"), error_analysis(gold, pred_gold_bert, "Transformer")])
errs.to_csv("error_analysis.csv", index=False, encoding="utf-8-sig")
print(errs.groupby(["model", "error", "type"]).size().unstack(fill_value=0))
errs[errs.model == "CRF"].sample(min(15, (errs.model == "CRF").sum()), random_state=1)[["error", "type", "gold", "pred"]]

## 10) Example input / output

In [ ]:
def extract_entities(text):
    doc = nlp(clean_body(text)); out = []
    for sent in doc.sents:
        r = sent_to_rec(sent)
        if len(r["tokens"]) < 2: continue
        tags = crf.predict_single(sent2features(r))
        for typ, s, e in get_entities(tags):
            a, b = doc[r["idx"][s]], doc[r["idx"][e]]
            out.append({"text": doc.text[a.idx:b.idx + len(b)], "label": typ, "start": a.idx, "end": b.idx + len(b)})
    return out

samples = ["Hi John Smith, please send invoice #INV-20451 to Enron Corp in Houston by March 3, 2001 for the natural gas contract.",
           df["body"].iloc[0], df["body"].iloc[1]]
examples = [{"input": s, "entities": extract_entities(s)} for s in samples]
os.makedirs("examples", exist_ok=True)
json.dump(examples, open("examples/example_outputs.json", "w"), indent=2, ensure_ascii=False)
examples[0]

In [ ]:
# log artifacts في MLflow
with mlflow.start_run(run_name="artifacts"):
    for f in ["error_analysis.csv", "examples/example_outputs.json"]: mlflow.log_artifact(f)
# لعرض الـ UI:  mlflow ui   (من الـ terminal في نفس فولدر المشروع)